## Simple Neural Network

In [1]:
import numpy as np

from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score

# Activation Function

def relu(z):
    return np.maximum(0, z)

def relu_derivative(z):
    return (z>0).astype(float)

def softmax(z):
    # Mumerical stability trick
    exp_z=np.exp(z-np.max(z, axis=1, keepdims=True))
    
    return exp_z/np.sum(exp_z, axis=1, keepdims=True)

# One Hot Encoding
def one_hot(y, n_classes):
    onehot=np.zeros((len(y), n_classes))
    onehot[np.arange(len(y)), y]=1
    return onehot


## Neural Network
class NN:
    def __init__(self, input_size, hidden_size, output_size, lr=0.01, epochs=1000):
        self.lr=lr
        self.epochs=epochs
        
        # Input -> Hidden
        self.W1=np.random.randn(input_size, hidden_size)*0.01
        self.b1=np.zeros((1, hidden_size))
        
        #hidden -> output
        self.W2=np.random.randn(
            hidden_size, 
            output_size
        )*0.01
        
        self.b2=np.zeros((1, output_size))
        
        
    # Forward Propagation
    def forward(self, X):
        # Input -> Hidden
        self.Z1=X@self.W1+self.b1
        self.A1=relu(self.Z1)
        
        # Hidden->Output
        self.Z2=self.A1@self.W2+self.b2
        self.A2=softmax(self.Z2)
        
        return self.A2
    
    
    # Loss function
    def compute_loss(self, y_true, y_pred):
        n=y_true.shape[0]
        loss=-np.sum(
            y_true*np.log(y_pred+1e-8)
        )/n
        
        return loss
    
    
    
    # BACKPROPAGATION
    def backward(self, X, y_true):
        n=X.shape[0]
        
        #output layer
        dZ2=self.A2-y_true
        
        dW2=(self.A1.T@ dZ2)/n
        
        db2=np.sum(
            dZ2, axis=0, keepdims=True
        )/n
        
        # Hidden layer
        dA1=dZ2 @ self.W2.T
        dZ1=dA1*relu_derivative(self.Z1)
        dW1=(X.T@dZ1)/n
        
        db1=np.sum(
            dZ1, axis=0,
            keepdims=True
        )/n
        
        # GRADIENT DESCENT
        self.W2 -= self.lr * dW2
        self.b2 -= self.lr * db2
        self.W1 -= self.lr * dW1
        self.b1 -= self.lr * db1
        
    
    def fit(self, X, y):
        n_classes=len(np.unique(y))
        
        y_encoded=one_hot(y, n_classes)
        
        for epoch in range(self.epochs):
            # Forward
            y_pred=self.forward(X)
            # Loss
            loss=self.compute_loss(y_encoded, y_pred)
            
            # Backward
            self.backward(X, y_encoded)
            
            #Print loss
            if epoch%100==0:
                print(f"Epoch {epoch}, Loss: {loss:.4f}")
                
    def predict(self, X):
        probs=self.forward(X)
        return np.argmax(probs, axis=1)
    

In [2]:
# Load Dataset
data=load_iris()

X=data.data
y=data.target

# Train Test Split
X_train, X_test, y_train, y_test=train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Scaling
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)


# My Model
model = NN(
    input_size=4,
    hidden_size=5,
    output_size=3,
    lr=0.01,
    epochs=1000
)

model.fit(X_train, y_train)

preds=model.predict(X_test)

# Accuracy
print("Accuracy:", accuracy_score(y_test, preds))

Epoch 0, Loss: 1.0986
Epoch 100, Loss: 1.0979
Epoch 200, Loss: 1.0962
Epoch 300, Loss: 1.0904
Epoch 400, Loss: 1.0700
Epoch 500, Loss: 1.0085
Epoch 600, Loss: 0.8883
Epoch 700, Loss: 0.7740
Epoch 800, Loss: 0.7025
Epoch 900, Loss: 0.6562
Accuracy: 0.6333333333333333
